# LLM Sampling Parameters - using Numpy - No call to LLM

* Temperature, Top-k, and Top-p — explained with inputs and outputs

---

When an LLM generates text, it does **not** always pick the single most likely word.
Instead it assigns a probability to every possible next word and then **samples** from those probabilities.

Three parameters control that sampling:

| Parameter | One-line summary |
|-----------|------------------|
| **Temperature** | Makes the distribution sharper (safe) or flatter (creative) |
| **Top-k** | Only keep the k most probable words |
| **Top-p** | Only keep words until their probabilities add up to p |

Every code cell below:
1. Defines a clear **input** (a probability distribution)
2. Applies **one parameter**
3. Prints the **output** so you can see exactly what changed

> **Requirements:** only the Python standard library + `numpy`

---
## Setup — helper functions and base data

In [1]:
import numpy as np

# ── Helper: pretty-print a word distribution ──────────────────────────────
def show(words, probs, title=''):
    """Print a word distribution as a simple text bar chart."""
    if title:
        print(title)
        print('-' * len(title))
    for w, p in zip(words, probs):
        bar    = '#' * int(round(p * 60))
        status = '  [kept]' if p > 0 else '  [removed]'
        print(f'  {w:<14} {p*100:5.1f}%  {bar}{status}')
    kept = sum(1 for p in probs if p > 0)
    print(f'  ({kept} of {len(probs)} words in the pool)\n')


# ── Base scenario ──────────────────────────────────────────────────────────
# Prompt: 'The chef's favourite dessert is ___'
# These are the model's raw output probabilities BEFORE any filter.

WORDS = ['chocolate', 'vanilla', 'strawberry', 'pizza', 'the', 'elephant', 'rocket']
BASE  = np.array([0.40, 0.30, 0.15, 0.08, 0.04, 0.02, 0.01])

show(WORDS, BASE, 'BASE DISTRIBUTION (no filter applied)')
print('Done')

BASE DISTRIBUTION (no filter applied)
-------------------------------------
  chocolate       40.0%  ########################  [kept]
  vanilla         30.0%  ##################  [kept]
  strawberry      15.0%  #########  [kept]
  pizza            8.0%  #####  [kept]
  the              4.0%  ##  [kept]
  elephant         2.0%  #  [kept]
  rocket           1.0%  #  [kept]
  (7 of 7 words in the pool)

Done


---
## Part 1 — Temperature

**What it does:**  
Temperature divides the log-probabilities before renormalising.

- `T < 1` → top word gets an even bigger share → **predictable**
- `T = 1` → probabilities stay exactly as they are
- `T > 1` → probabilities get closer together → **creative / risky**

**Analogy:** a volume knob — turn down for safe, turn up for wild.

In [2]:
def apply_temperature(probs, T):
    """Rescale a probability distribution by temperature T."""
    log_p  = np.log(np.clip(probs, 1e-10, None)) / T
    log_p -= log_p.max()          # subtract max for numerical stability
    exp_p  = np.exp(log_p)
    return exp_p / exp_p.sum()


# ── Input ──────────────────────────────────────────────────────────────────
show(WORDS, BASE, 'INPUT (base distribution)')

# ── Output at different temperatures ──────────────────────────────────────
for T in [0.2, 0.5, 1.0, 1.5, 2.0]:
    out = apply_temperature(BASE, T)
    show(WORDS, out, f'OUTPUT  temperature = {T}')

INPUT (base distribution)
-------------------------
  chocolate       40.0%  ########################  [kept]
  vanilla         30.0%  ##################  [kept]
  strawberry      15.0%  #########  [kept]
  pizza            8.0%  #####  [kept]
  the              4.0%  ##  [kept]
  elephant         2.0%  #  [kept]
  rocket           1.0%  #  [kept]
  (7 of 7 words in the pool)

OUTPUT  temperature = 0.2
-------------------------
  chocolate       80.3%  ################################################  [kept]
  vanilla         19.1%  ###########  [kept]
  strawberry       0.6%    [kept]
  pizza            0.0%    [kept]
  the              0.0%    [kept]
  elephant         0.0%    [kept]
  rocket           0.0%    [kept]
  (7 of 7 words in the pool)

OUTPUT  temperature = 0.5
-------------------------
  chocolate       56.9%  ##################################  [kept]
  vanilla         32.0%  ###################  [kept]
  strawberry       8.0%  #####  [kept]
  pizza            2.3%  #  [

### What to notice

- At **T = 0.2** almost all the probability collapses onto `chocolate`.  
  The model will almost always say `chocolate` — very predictable.
- At **T = 1.0** the numbers are identical to the input — no change.
- At **T = 2.0** the gap between words narrows dramatically.  
  `elephant` and `rocket` now have a real chance of being picked — creative but risky.

### When to use which temperature

| Use case | Recommended T |
|----------|---------------|
| Maths, code, factual Q&A | 0.0 – 0.3 |
| Summarisation, translation | 0.3 – 0.7 |
| Creative writing | 0.7 – 1.2 |
| Poetry, experimental fiction | 1.2 – 2.0 |

---
## Part 2 — Top-k

**What it does:**  
Keep only the **k most probable words**. Discard the rest entirely.
Then renormalise the survivors so they sum to 1.

**Analogy:** a job shortlist — 'I will only interview the top 3 candidates,
no matter how close or far apart they are.'

In [3]:
def apply_top_k(probs, k):
    """Zero out everything outside the top-k words, then renormalise."""
    k        = max(1, min(k, len(probs)))
    filtered = probs.copy()
    filtered[k:] = 0.0
    filtered /= filtered.sum()
    return filtered


# ── Input ──────────────────────────────────────────────────────────────────
show(WORDS, BASE, 'INPUT (base distribution)')

# ── Output at different k values ───────────────────────────────────────────
for k in [1, 2, 3, 5, 7]:
    out = apply_top_k(BASE, k)
    show(WORDS, out, f'OUTPUT  top-k = {k}')

INPUT (base distribution)
-------------------------
  chocolate       40.0%  ########################  [kept]
  vanilla         30.0%  ##################  [kept]
  strawberry      15.0%  #########  [kept]
  pizza            8.0%  #####  [kept]
  the              4.0%  ##  [kept]
  elephant         2.0%  #  [kept]
  rocket           1.0%  #  [kept]
  (7 of 7 words in the pool)

OUTPUT  top-k = 1
-----------------
  chocolate      100.0%  ############################################################  [kept]
  vanilla          0.0%    [removed]
  strawberry       0.0%    [removed]
  pizza            0.0%    [removed]
  the              0.0%    [removed]
  elephant         0.0%    [removed]
  rocket           0.0%    [removed]
  (1 of 7 words in the pool)

OUTPUT  top-k = 2
-----------------
  chocolate       57.1%  ##################################  [kept]
  vanilla         42.9%  ##########################  [kept]
  strawberry       0.0%    [removed]
  pizza            0.0%    [removed]


### What to notice

- **k = 1**: only `chocolate` survives. The model is forced to pick it every time (greedy decoding).
- **k = 3**: `chocolate`, `vanilla`, `strawberry` are in the pool. `pizza` and beyond are invisible.
- **k = 7**: all words survive (same as no filter) — but their probabilities are renormalised.

### The problem with top-k
Top-k is a **fixed headcount** — it ignores how confident the model actually is:
- If one word has 95% probability, k=10 still forces 9 extra near-zero candidates in.
- If 15 words share the probability evenly, k=10 unfairly cuts 5 of them out.

This is exactly the problem top-p solves.

---
## Part 3 — Top-p (Nucleus Sampling)

**What it does:**  
Add up probabilities from highest to lowest.  
Stop as soon as the running total reaches **p**.  
Keep only those words (the 'nucleus'). Discard the rest. Renormalise.

**Analogy:** a party guest list — 'keep inviting people (most popular first)
until the combined popularity of the guests covers 90% of the crowd.'

**Key advantage over top-k:** the pool size adapts automatically.  
Confident model → small nucleus. Uncertain model → large nucleus.

In [4]:
def apply_top_p(probs, p):
    """Keep the fewest top words whose cumulative probability reaches p."""
    cumulative  = np.cumsum(probs)
    # find the index where cumulative sum first reaches p
    nucleus_end = int(np.searchsorted(cumulative, p, side='left')) + 1
    nucleus_end = max(1, min(nucleus_end, len(probs)))
    filtered    = probs.copy()
    filtered[nucleus_end:] = 0.0
    filtered   /= filtered.sum()
    return filtered


# ── Input ──────────────────────────────────────────────────────────────────
show(WORDS, BASE, 'INPUT (base distribution)')

# ── Output at different p values ───────────────────────────────────────────
for p in [0.40, 0.70, 0.85, 0.95, 1.00]:
    out = apply_top_p(BASE, p)
    show(WORDS, out, f'OUTPUT  top-p = {p}')

INPUT (base distribution)
-------------------------
  chocolate       40.0%  ########################  [kept]
  vanilla         30.0%  ##################  [kept]
  strawberry      15.0%  #########  [kept]
  pizza            8.0%  #####  [kept]
  the              4.0%  ##  [kept]
  elephant         2.0%  #  [kept]
  rocket           1.0%  #  [kept]
  (7 of 7 words in the pool)

OUTPUT  top-p = 0.4
-------------------
  chocolate      100.0%  ############################################################  [kept]
  vanilla          0.0%    [removed]
  strawberry       0.0%    [removed]
  pizza            0.0%    [removed]
  the              0.0%    [removed]
  elephant         0.0%    [removed]
  rocket           0.0%    [removed]
  (1 of 7 words in the pool)

OUTPUT  top-p = 0.7
-------------------
  chocolate       57.1%  ##################################  [kept]
  vanilla         42.9%  ##########################  [kept]
  strawberry       0.0%    [removed]
  pizza            0.0%    [r

### What to notice

- **p = 0.40**: only `chocolate` survives (it alone already covers 40%).
- **p = 0.70**: `chocolate` + `vanilla` together cover 70% → 2 words.
- **p = 0.85**: `chocolate` + `vanilla` + `strawberry` cover 85% → 3 words.
- **p = 1.00**: all words survive (no filter).

The nucleus size grows *as needed* — that is the key insight.

---
## Part 4 — Top-p vs Top-k: The Critical Difference

The same settings behave very differently depending on how confident the model is.
We use two extreme scenarios to show this.

In [5]:
# Scenario A: Model is VERY CONFIDENT
# Context: 'The capital of France is ___'
# One answer dominates strongly.

WORDS_A = ['Paris', 'Lyon', 'Rome', 'Berlin', 'Madrid', 'Oslo', 'Cairo']
PROBS_A = np.array([0.85, 0.08, 0.03, 0.02, 0.01, 0.005, 0.005])

print('SCENARIO A: Very confident model')
print('Context: "The capital of France is ___"\n')
show(WORDS_A, PROBS_A, 'Input')
show(WORDS_A, apply_top_k(PROBS_A, 3),    'top-k=3   (always keeps exactly 3)')
show(WORDS_A, apply_top_p(PROBS_A, 0.90), 'top-p=0.90 (adapts to confidence)')

SCENARIO A: Very confident model
Context: "The capital of France is ___"

Input
-----
  Paris           85.0%  ###################################################  [kept]
  Lyon             8.0%  #####  [kept]
  Rome             3.0%  ##  [kept]
  Berlin           2.0%  #  [kept]
  Madrid           1.0%  #  [kept]
  Oslo             0.5%    [kept]
  Cairo            0.5%    [kept]
  (7 of 7 words in the pool)

top-k=3   (always keeps exactly 3)
----------------------------------
  Paris           88.5%  #####################################################  [kept]
  Lyon             8.3%  #####  [kept]
  Rome             3.1%  ##  [kept]
  Berlin           0.0%    [removed]
  Madrid           0.0%    [removed]
  Oslo             0.0%    [removed]
  Cairo            0.0%    [removed]
  (3 of 7 words in the pool)

top-p=0.90 (adapts to confidence)
---------------------------------
  Paris           91.4%  #######################################################  [kept]
  Lyon             

In [6]:
# Scenario B: Model is VERY UNCERTAIN
# Context: 'The best thing to do on a Sunday is ___'
# Many answers are equally plausible.

WORDS_B = ['read', 'cook', 'hike', 'game', 'sleep', 'paint', 'garden']
PROBS_B = np.array([0.18, 0.17, 0.16, 0.15, 0.14, 0.11, 0.09])

print('SCENARIO B: Very uncertain model')
print('Context: "The best thing to do on a Sunday is ___"\n')
show(WORDS_B, PROBS_B, 'Input')
show(WORDS_B, apply_top_k(PROBS_B, 3),    'top-k=3   (always keeps exactly 3)')
show(WORDS_B, apply_top_p(PROBS_B, 0.90), 'top-p=0.90 (adapts to confidence)')

SCENARIO B: Very uncertain model
Context: "The best thing to do on a Sunday is ___"

Input
-----
  read            18.0%  ###########  [kept]
  cook            17.0%  ##########  [kept]
  hike            16.0%  ##########  [kept]
  game            15.0%  #########  [kept]
  sleep           14.0%  ########  [kept]
  paint           11.0%  #######  [kept]
  garden           9.0%  #####  [kept]
  (7 of 7 words in the pool)

top-k=3   (always keeps exactly 3)
----------------------------------
  read            35.3%  #####################  [kept]
  cook            33.3%  ####################  [kept]
  hike            31.4%  ###################  [kept]
  game             0.0%    [removed]
  sleep            0.0%    [removed]
  paint            0.0%    [removed]
  garden           0.0%    [removed]
  (3 of 7 words in the pool)

top-p=0.90 (adapts to confidence)
---------------------------------
  read            19.8%  ############  [kept]
  cook            18.7%  ###########  [kept]
  hike

### Summary of the comparison

| | top-k = 3 | top-p = 0.90 |
|--|-----------|-------------|
| Confident model (Paris 85%) | keeps 3 words — wasteful, 2 wrong answers pollute the pool | keeps 1 word — correctly locks onto `Paris` |
| Uncertain model (all ~15%) | keeps 3 words — too narrow, cuts 4 equally valid options | keeps 6-7 words — gives all valid options a fair chance |

**Top-k is a blunt instrument. Top-p is context-aware.**

---
## Part 5 — All Three Together: The Full Pipeline

In a real LLM all three filters are applied **in sequence**:

```
Base probabilities
  -> Temperature  (reshape the curve)
  -> Top-k        (hard word-count cut)
  -> Top-p        (adaptive probability cut)
  -> Sample       (pick one word randomly from survivors)
```

In [7]:
def full_pipeline(probs, words, temperature=1.0, top_k=None, top_p=None):
    """
    Run the full LLM sampling pipeline and print each step.

    Parameters
    ----------
    probs       : base probability array (sums to 1)
    words       : word labels
    temperature : float  (default 1.0 = no change)
    top_k       : int or None
    top_p       : float or None

    Returns
    -------
    The sampled word (str)
    """
    p = probs.copy()
    show(words, p, 'STEP 0 — Base distribution')

    # Step 1: Temperature
    p = apply_temperature(p, temperature)
    show(words, p, f'STEP 1 — After temperature = {temperature}')

    # Step 2: Top-k
    if top_k is not None:
        p = apply_top_k(p, top_k)
        show(words, p, f'STEP 2 — After top-k = {top_k}')
    else:
        print('STEP 2 — top-k: disabled\n')

    # Step 3: Top-p
    if top_p is not None:
        p = apply_top_p(p, top_p)
        show(words, p, f'STEP 3 — After top-p = {top_p}')
    else:
        print('STEP 3 — top-p: disabled\n')

    # Step 4: Sample
    survivors = [(w, prob) for w, prob in zip(words, p) if prob > 0]
    s_words   = [w for w, _ in survivors]
    s_probs   = np.array([prob for _, prob in survivors])
    chosen    = np.random.choice(s_words, p=s_probs)
    print(f'STEP 4 — Sample from {s_words}')
    print(f'         => picked: "{chosen}"')
    return chosen


print('=' * 60)
print('EXAMPLE PIPELINE  temperature=0.8, top_k=5, top_p=0.92')
print('=' * 60)
full_pipeline(BASE, WORDS, temperature=0.8, top_k=5, top_p=0.92)

EXAMPLE PIPELINE  temperature=0.8, top_k=5, top_p=0.92
STEP 0 — Base distribution
--------------------------
  chocolate       40.0%  ########################  [kept]
  vanilla         30.0%  ##################  [kept]
  strawberry      15.0%  #########  [kept]
  pizza            8.0%  #####  [kept]
  the              4.0%  ##  [kept]
  elephant         2.0%  #  [kept]
  rocket           1.0%  #  [kept]
  (7 of 7 words in the pool)

STEP 1 — After temperature = 0.8
--------------------------------
  chocolate       45.1%  ###########################  [kept]
  vanilla         31.5%  ###################  [kept]
  strawberry      13.2%  ########  [kept]
  pizza            6.0%  ####  [kept]
  the              2.5%  ##  [kept]
  elephant         1.1%  #  [kept]
  rocket           0.4%    [kept]
  (7 of 7 words in the pool)

STEP 2 — After top-k = 5
------------------------
  chocolate       45.8%  ############################  [kept]
  vanilla         32.0%  ###################  [kept]
  s

np.str_('strawberry')

---
## Part 6 — Experiment: try your own settings

Change the values in the cell below and run it to see how the output changes.

In [ ]:
# ── Change these values and re-run ───────────────────────────────────────
MY_TEMPERATURE = 1.0    # Try: 0.1, 0.5, 1.0, 1.5, 2.0
MY_TOP_K       = 4      # Try: 1, 2, 3, 5, None (to disable)
MY_TOP_P       = 0.90   # Try: 0.40, 0.70, 0.85, 0.95, 1.0, None (to disable)
# ─────────────────────────────────────────────────────────────────────────

full_pipeline(BASE, WORDS,
              temperature=MY_TEMPERATURE,
              top_k=MY_TOP_K,
              top_p=MY_TOP_P)

In [ ]:
# ── Run many samples to see the distribution ─────────────────────────────
# Change settings and run to see how frequently each word gets picked.

MY_TEMPERATURE = 1.0
MY_TOP_K       = 4
MY_TOP_P       = 0.90
N_SAMPLES      = 50     # number of words to sample

from collections import Counter

def sample_once(probs, words, temperature, top_k, top_p):
    p = apply_temperature(probs, temperature)
    if top_k is not None:
        p = apply_top_k(p, top_k)
    if top_p is not None:
        p = apply_top_p(p, top_p)
    survivors = [(w, prob) for w, prob in zip(words, p) if prob > 0]
    s_words   = [w for w, _ in survivors]
    s_probs   = np.array([prob for _, prob in survivors])
    s_probs  /= s_probs.sum()
    return np.random.choice(s_words, p=s_probs)

results = [sample_once(BASE, WORDS, MY_TEMPERATURE, MY_TOP_K, MY_TOP_P)
           for _ in range(N_SAMPLES)]

counts = Counter(results)

print(f'{N_SAMPLES} samples  (T={MY_TEMPERATURE}, k={MY_TOP_K}, p={MY_TOP_P})')
print('-' * 50)
for w in WORDS:
    c   = counts.get(w, 0)
    bar = '#' * c
    print(f'  {w:<14} {c:>3}x  {bar}')

---
## Quick Reference

### Temperature
| Value | Effect | Good for |
|-------|--------|----------|
| 0.0 – 0.3 | Near-deterministic | Code, maths, facts |
| 0.4 – 0.7 | Natural, balanced | Summaries, email |
| 0.8 – 1.2 | Creative, varied | Chat, storytelling |
| 1.3 – 2.0 | Wild, experimental | Poetry, brainstorming |

### Top-k
| Value | Effect |
|-------|--------|
| k = 1 | Greedy decoding — always picks the top word |
| k = 5–20 | Tight control — good for structured outputs |
| k = 40–100 | General use — variety without noise |
| k = None | Disabled — let top-p handle the filtering |

### Top-p
| Value | Effect |
|-------|--------|
| p = 0.1–0.5 | Very focused, small nucleus |
| p = 0.7–0.9 | Sweet spot — adaptive, coherent |
| p = 0.95–1.0 | Wide open — includes rarer words |

### Typical presets
| Task | Temperature | Top-k | Top-p |
|------|------------|-------|-------|
| Code / facts | 0.2 | 10 | 0.9 |
| General chat | 0.8 | 50 | 0.9 |
| Creative writing | 1.0 | 100 | 0.95 |
| Experimental | 1.5 | None | 1.0 |